# 第八节课课件：辛积分方法——从双摆到结构保持数值算法

> 课程定位：前几节我们把物理规律写进 PINN 的 loss；第 7 节我们看到 Hamilton 系统天然带有辛结构。本节课把这两条线合起来：数值方法不仅要“算得准”，还要尽量保持问题本身的重要结构。

本节目标：

1. 观察普通 RK4 在双摆长时间积分中的 Hamiltonian 漂移；
2. 理解辛结构为什么是 Hamilton 系统的核心几何约束；
3. 实现并比较三类辛积分器：Symplectic Euler、Störmer-Verlet、Implicit Midpoint；
4. 用能量漂移、相图和双摆末端轨迹评价长时间数值行为；
5. 思考如何把“守恒约束 PINN”和“辛积分”统一到结构保持方法的论文主题中。

给学生的一句话总结：**本节课讨论的不是更花哨的时间步进，而是如何让数值算法尊重物理系统的几何结构。**


# 1. 回顾与定位：两类结构保持

前面课程中我们已经接触了两类“结构保持”思想。

第一类来自 PDE-PINN 线：

- 热传导方程中，PINN 把 PDE 残差写进 loss；
- 线性平流方程和 Burgers 方程中，我们进一步把质量、动量等守恒量写进 loss；
- 这些方法强调：神经网络不能只拟合数据，还要满足物理约束。

第二类来自 Hamilton 系统线：

- 第 7 节的双摆写成正则变量 $z=(q,p)$；
- Hamilton 方程为

$$
\dot z = J\nabla H(z),
\qquad
J=\begin{bmatrix}0&I\\-I&0\end{bmatrix};
$$

- 正则变换不是普通变量替换，它保持 Hamilton 方程的辛结构。

因此本课程的核心线索可以写成：

$$
\text{物理守恒约束} \quad \Longleftrightarrow \quad \text{几何结构约束}.
$$

在 PINN 中，我们常把结构写进 loss；在数值积分中，我们把结构写进算法本身。

给学生的一句话总结：**PDE 守恒量和 Hamilton 辛结构看起来不同，本质上都在回答同一个问题：算法是否尊重物理问题本身。**


# 2. 为什么 RK4 不够好？

RK4 是非常经典、非常好用的通用 ODE 求解器。它的局部截断误差阶数高，短时间精度通常很好。

但对于 Hamilton 系统，仅仅“短时间误差小”还不够。Hamilton 系统具有几何结构：

$$
\omega = dq_1\wedge dp_1 + dq_2\wedge dp_2, dq(t)\wedge dp(t) = dq(0)\wedge dp(0)
$$

真实流映射会保持这个辛二形式。普通 RK4 并不是辛积分器，因此它不会严格保持这个几何结构。长时间积分时，Hamiltonian 可能出现系统性漂移。

本节先复用第 7 节的双摆 Hamiltonian，并用 PyTorch 自动微分计算右端项。

给学生的一句话总结：**RK4 很强，但它不是为 Hamilton 系统的长期几何性质设计的。**


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Callable, Dict, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch

# 本节所有实验都在 CPU 和 float64 上运行。
# 双摆的能量漂移通常很小，float64 更适合观察长期行为。
torch.set_default_dtype(torch.float64)
np.set_printoptions(precision=4, suppress=True)

@dataclass
class Parameters:
    m1: float = 1.0
    m2: float = 1.0
    l1: float = 1.0
    l2: float = 1.0
    g: float = 9.81

params = Parameters()


## 2.1 双摆 Hamiltonian

沿用第 7 节的正则变量：

$$
z=(q_1,q_2,p_1,p_2)^T,
$$

其中 $q_1,q_2$ 是两根摆杆相对竖直方向的角度，$p_1,p_2$ 是共轭动量。记

$$
\Delta=q_1-q_2.
$$

双摆 Hamiltonian 为

$$
H(q,p)=
\frac{
 m_2l_2^2p_1^2+(m_1+m_2)l_1^2p_2^2
 -2m_2l_1l_2\cos\Delta\,p_1p_2
}{
2m_2l_1^2l_2^2\left(m_1+m_2\sin^2\Delta\right)
}
-(m_1+m_2)gl_1\cos q_1-m_2gl_2\cos q_2.
$$

Hamilton 正则方程为

$$
\dot q=\frac{\partial H}{\partial p},
\qquad
\dot p=-\frac{\partial H}{\partial q}.
$$

给学生的一句话总结：**只要能写出 Hamiltonian，PyTorch 自动微分就可以帮我们得到 Hamilton 方程右端项。**


In [ ]:
def hamiltonian(z: torch.Tensor, params: Parameters = params) -> torch.Tensor:
    """Double-pendulum Hamiltonian H(q1, q2, p1, p2).

    z can be shape (4,) or (..., 4).
    """
    q1, q2, p1, p2 = z.unbind(dim=-1)
    delta = q1 - q2
    denominator = (
        2.0
        * params.m2
        * params.l1**2
        * params.l2**2
        * (params.m1 + params.m2 * torch.sin(delta) ** 2)
    )
    numerator = (
        params.m2 * params.l2**2 * p1**2
        + (params.m1 + params.m2) * params.l1**2 * p2**2
        - 2.0
        * params.m2
        * params.l1
        * params.l2
        * torch.cos(delta)
        * p1
        * p2
    )
    potential = (
        -(params.m1 + params.m2) * params.g * params.l1 * torch.cos(q1)
        - params.m2 * params.g * params.l2 * torch.cos(q2)
    )
    return numerator / denominator + potential


def grad_hamiltonian_np(state: np.ndarray) -> np.ndarray:
    """Return grad_z H as a NumPy array using PyTorch autograd."""
    z = torch.tensor(state, dtype=torch.float64, device="cpu", requires_grad=True)
    energy = hamiltonian(z)
    grad_h = torch.autograd.grad(energy, z)[0]
    return grad_h.detach().numpy()


def hamiltonian_rhs(state: np.ndarray) -> np.ndarray:
    """Return dz/dt = J grad H = (H_p, -H_q)."""
    grad_h = grad_hamiltonian_np(state)
    return np.concatenate([grad_h[2:], -grad_h[:2]])


def split_grad_np(q: np.ndarray, p: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Return (H_q, H_p) for q=(q1,q2), p=(p1,p2)."""
    z = np.concatenate([q, p])
    grad_h = grad_hamiltonian_np(z)
    return grad_h[:2], grad_h[2:]


def energy_history(states: np.ndarray) -> np.ndarray:
    with torch.no_grad():
        tensor = torch.tensor(states, dtype=torch.float64, device="cpu")
        return hamiltonian(tensor).numpy()


## 2.2 RK4 长时间积分

RK4 的一步格式为

$$
\begin{aligned}
k_1 &= f(z_n),\\
k_2 &= f(z_n+\tfrac{\Delta t}{2}k_1),\\
k_3 &= f(z_n+\tfrac{\Delta t}{2}k_2),\\
k_4 &= f(z_n+\Delta t k_3),\\
z_{n+1} &= z_n+\frac{\Delta t}{6}(k_1+2k_2+2k_3+k_4).
\end{aligned}
$$

其中 $f(z)=J\nabla H(z)$。

下面先跑一个较长时间的 RK4，并画出

$$
|H(t)-H(0)|.
$$

给学生的一句话总结：**能量误差不是唯一指标，但它是观察 Hamilton 系统长期质量的第一扇窗。**


In [ ]:
def rk4_step(state: np.ndarray, dt: float) -> np.ndarray:
    k1 = hamiltonian_rhs(state)
    k2 = hamiltonian_rhs(state + 0.5 * dt * k1)
    k3 = hamiltonian_rhs(state + 0.5 * dt * k2)
    k4 = hamiltonian_rhs(state + dt * k3)
    return state + dt * (k1 + 2.0 * k2 + 2.0 * k3 + k4) / 6.0


def integrate(stepper: Callable[[np.ndarray, float], np.ndarray], z0: np.ndarray, t_end: float, dt: float) -> Tuple[np.ndarray, np.ndarray]:
    steps = int(np.ceil(t_end / dt))
    times = np.linspace(0.0, steps * dt, steps + 1)
    states = np.empty((steps + 1, len(z0)), dtype=np.float64)
    states[0] = z0
    for n in range(steps):
        states[n + 1] = stepper(states[n], dt)
    return times, states


def plot_energy_drift(times: np.ndarray, states: np.ndarray, label: str = "RK4") -> None:
    energies = energy_history(states)
    drift = np.abs(energies - energies[0])
    plt.figure(figsize=(8, 4))
    plt.semilogy(times, drift + 1e-18, label=label)
    plt.xlabel("t [s]")
    plt.ylabel(r"$|H(t)-H(0)|$")
    plt.title("Hamiltonian drift")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()

z0 = np.array([1.0, 0.2, 0.0, 0.0], dtype=np.float64)

times_rk4, states_rk4 = integrate(rk4_step, z0, t_end=30.0, dt=0.01)
plot_energy_drift(times_rk4, states_rk4, "RK4, dt=0.01")
print("max drift:", np.max(np.abs(energy_history(states_rk4) - energy_history(states_rk4)[0])))


# 3. 辛积分的数学直觉

Hamilton 系统的真实时间推进可以看成一个映射：

$$
\Phi_{\Delta t}: z_n \mapsto z_{n+1}.
$$

如果这个映射保持辛二形式

$$
\omega = dq\wedge dp,
$$

就称它是辛映射。用矩阵直觉表示，如果 $D\Phi$ 是映射的 Jacobian，那么辛条件可以写成

$$
(D\Phi)^T J (D\Phi)=J.
$$

这条公式可以直观理解为：相空间中的位置和动量不是普通坐标，它们成对出现；辛映射保持这种配对关系。

Liouville 定理告诉我们，Hamilton 系统的真实流会保持相空间体积。辛积分器可以看成这个性质的离散类比：每一步时间推进都尽量保持 Hamilton 系统的几何结构。

一个常见现象是：

- 非辛方法可能表现出数值耗散或数值增能，能量误差长期偏离；
- 辛方法不一定每一步都精确守恒 Hamiltonian，但能量误差通常在一个范围内振荡，不容易出现长期单调漂移。

给学生的一句话总结：**辛积分不是保证能量一动不动，而是让长期轨道活在更接近真实 Hamilton 系统的几何世界里。**


# 4. 三种辛积分器

下面实现三种辛积分器。

需要特别说明：双摆 Hamiltonian 不是简单可分离形式

$$
H(q,p)\ne T(p)+V(q),
$$

因为动能项含有 $q_1-q_2$，也就是质量矩阵依赖坐标。因此经典“显式速度 Verlet”不能直接照搬。我们使用适用于一般 Hamiltonian 的半隐式或隐式形式，并用固定点迭代求解每一步里的隐式方程。

这些固定点迭代不是本节重点。你只需要知道：隐式方程的目标是让数值格式满足辛结构，而不是单纯为了增加代码复杂度。

给学生的一句话总结：**对非线性、非分离 Hamiltonian，辛积分经常需要每一步解一个小的隐式问题。**


In [ ]:
def fixed_point_solve(
    update: Callable[[np.ndarray], np.ndarray],
    guess: np.ndarray,
    max_iter: int = 30,
    tol: float = 1e-12,
) -> np.ndarray:
    """Simple fixed-point iteration for the small implicit systems in this lesson."""
    x = guess.astype(np.float64).copy()
    for _ in range(max_iter):
        x_new = update(x)
        if np.linalg.norm(x_new - x, ord=np.inf) < tol:
            return x_new
        x = x_new
    return x


def symplectic_euler_step(state: np.ndarray, dt: float) -> np.ndarray:
    """Semi-implicit symplectic Euler for a general Hamiltonian.

    p_{n+1} = p_n - dt * H_q(q_n, p_{n+1})
    q_{n+1} = q_n + dt * H_p(q_n, p_{n+1})
    """
    q_n = state[:2]
    p_n = state[2:]

    def update_p(p_guess: np.ndarray) -> np.ndarray:
        h_q, _ = split_grad_np(q_n, p_guess)
        return p_n - dt * h_q

    p_next = fixed_point_solve(update_p, p_n)
    _, h_p = split_grad_np(q_n, p_next)
    q_next = q_n + dt * h_p
    return np.concatenate([q_next, p_next])


## 4.1 Symplectic Euler（一阶）

本节采用的 Symplectic Euler 格式为

$$
p_{n+1}=p_n-\Delta t\,\frac{\partial H}{\partial q}(q_n,p_{n+1}),
$$

$$
q_{n+1}=q_n+\Delta t\,\frac{\partial H}{\partial p}(q_n,p_{n+1}).
$$

它只有一阶精度，但已经是辛格式。第一行中 $p_{n+1}$ 出现在右端，因此对一般 Hamiltonian 是隐式的。

给学生的一句话总结：**Symplectic Euler 精度不高，但它用最小代价展示了“算法结构比阶数更重要”的思想。**


In [ ]:
def stormer_verlet_step(state: np.ndarray, dt: float) -> np.ndarray:
    """Generalized implicit Störmer-Verlet for nonseparable Hamiltonians.

    p_{n+1/2} = p_n - dt/2 * H_q(q_n, p_{n+1/2})
    q_{n+1} = q_n + dt/2 * [H_p(q_n, p_{n+1/2}) + H_p(q_{n+1}, p_{n+1/2})]
    p_{n+1} = p_{n+1/2} - dt/2 * H_q(q_{n+1}, p_{n+1/2})
    """
    q_n = state[:2]
    p_n = state[2:]

    def update_p_half(p_guess: np.ndarray) -> np.ndarray:
        h_q, _ = split_grad_np(q_n, p_guess)
        return p_n - 0.5 * dt * h_q

    p_half = fixed_point_solve(update_p_half, p_n)
    _, h_p_left = split_grad_np(q_n, p_half)

    def update_q(q_guess: np.ndarray) -> np.ndarray:
        _, h_p_right = split_grad_np(q_guess, p_half)
        return q_n + 0.5 * dt * (h_p_left + h_p_right)

    q_next = fixed_point_solve(update_q, q_n + dt * h_p_left)
    h_q_right, _ = split_grad_np(q_next, p_half)
    p_next = p_half - 0.5 * dt * h_q_right
    return np.concatenate([q_next, p_next])


## 4.2 Störmer-Verlet（二阶）

如果 Hamiltonian 可以分离为 $H(q,p)=T(p)+V(q)$，经典 Störmer-Verlet 可以写成非常熟悉的“半步动量、整步位置、半步动量”。

双摆不是可分离 Hamiltonian，因此我们使用广义隐式 Störmer-Verlet：

$$
p_{n+1/2}=p_n-\frac{\Delta t}{2}\frac{\partial H}{\partial q}(q_n,p_{n+1/2}),
$$

$$
q_{n+1}=q_n+\frac{\Delta t}{2}\left[
\frac{\partial H}{\partial p}(q_n,p_{n+1/2})+
\frac{\partial H}{\partial p}(q_{n+1},p_{n+1/2})
\right],
$$

$$
p_{n+1}=p_{n+1/2}-\frac{\Delta t}{2}\frac{\partial H}{\partial q}(q_{n+1},p_{n+1/2}).
$$

该格式是二阶辛格式，也具有时间反演对称性。

给学生的一句话总结：**Verlet 的核心不是某个固定代码模板，而是“半步-整步-半步”背后的辛结构。**


In [ ]:
def implicit_midpoint_step(state: np.ndarray, dt: float) -> np.ndarray:
    """Implicit midpoint rule, a second-order symplectic Runge-Kutta method.

    z_{n+1} = z_n + dt * J grad H((z_n + z_{n+1})/2)
    """
    z_n = state

    def update_z(z_guess: np.ndarray) -> np.ndarray:
        midpoint = 0.5 * (z_n + z_guess)
        return z_n + dt * hamiltonian_rhs(midpoint)

    # RK4 prediction gives a good initial guess for the fixed-point iteration.
    z_guess = rk4_step(z_n, dt)
    return fixed_point_solve(update_z, z_guess, max_iter=40, tol=1e-12)


## 4.3 Implicit Midpoint（二阶辛）

隐式中点法写成

$$
z_{n+1}=z_n+\Delta t\,J\nabla H\left(\frac{z_n+z_{n+1}}{2}\right).
$$

它是二阶 Runge-Kutta 方法，同时也是辛方法。和 RK4 相比，它的阶数更低，但几何性质更适合 Hamilton 系统长期模拟。

给学生的一句话总结：**隐式中点法说明：阶数不是唯一标准，辛结构有时比更高阶的局部精度更重要。**


# 5. 四种积分器对比实验

我们比较四种方法：

1. RK4；
2. Symplectic Euler；
3. Störmer-Verlet；
4. Implicit Midpoint。

核心指标包括：

- Hamiltonian 漂移曲线；
- 能量误差的 long-time behavior；
- 双摆第二个质点的末端轨迹；
- 状态变量相图，例如 $(q_1,p_1)$ 和 $(q_2,p_2)$；
- 不同时间步长下的最大能量漂移，$\Delta t=0.01,0.005,0.0025$。

给学生的一句话总结：**结构保持方法的价值通常要在长时间、多指标对比中才看得清楚。**


In [ ]:
STEPPERS: Dict[str, Callable[[np.ndarray, float], np.ndarray]] = {
    "RK4": rk4_step,
    "Symplectic Euler": symplectic_euler_step,
    "Störmer-Verlet": stormer_verlet_step,
    "Implicit Midpoint": implicit_midpoint_step,
}


def cartesian_positions(states: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    q1, q2 = states[:, 0], states[:, 1]
    x1 = params.l1 * np.sin(q1)
    y1 = -params.l1 * np.cos(q1)
    x2 = x1 + params.l2 * np.sin(q2)
    y2 = y1 - params.l2 * np.cos(q2)
    return x1, y1, x2, y2


def run_all_methods(z0: np.ndarray, t_end: float, dt: float) -> Dict[str, Tuple[np.ndarray, np.ndarray]]:
    results = {}
    for name, stepper in STEPPERS.items():
        print(f"running {name:>18s}, dt={dt}")
        results[name] = integrate(stepper, z0, t_end=t_end, dt=dt)
    return results


def summarize_energy(results: Dict[str, Tuple[np.ndarray, np.ndarray]]) -> None:
    print(f"{'method':>20s} | {'max |H-H0|':>14s} | {'final H-H0':>14s}")
    print("-" * 56)
    for name, (_, states) in results.items():
        e = energy_history(states)
        drift = e - e[0]
        print(f"{name:>20s} | {np.max(np.abs(drift)):14.6e} | {drift[-1]:14.6e}")

results_dt001 = run_all_methods(z0, t_end=30.0, dt=0.01)
summarize_energy(results_dt001)


In [ ]:
def plot_compare_energy(results: Dict[str, Tuple[np.ndarray, np.ndarray]], title: str) -> None:
    plt.figure(figsize=(9, 5))
    for name, (times, states) in results.items():
        e = energy_history(states)
        plt.semilogy(times, np.abs(e - e[0]) + 1e-18, label=name)
    plt.xlabel("t [s]")
    plt.ylabel(r"$|H(t)-H(0)|$")
    plt.title(title)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()

plot_compare_energy(results_dt001, "Hamiltonian drift, dt=0.01")


In [ ]:
def plot_tip_trajectories(results: Dict[str, Tuple[np.ndarray, np.ndarray]], title: str) -> None:
    fig, axes = plt.subplots(2, 2, figsize=(10, 9), sharex=True, sharey=True)
    axes = axes.ravel()
    for ax, (name, (_, states)) in zip(axes, results.items()):
        _, _, x2, y2 = cartesian_positions(states)
        ax.plot(x2, y2, lw=0.8)
        ax.set_title(name)
        ax.set_xlabel("x2")
        ax.set_ylabel("y2")
        ax.axis("equal")
        ax.grid(alpha=0.3)
    fig.suptitle(title)
    fig.tight_layout()

plot_tip_trajectories(results_dt001, "Trajectory of the second mass, dt=0.01")


In [ ]:
def plot_phase_portraits(results: Dict[str, Tuple[np.ndarray, np.ndarray]], title: str) -> None:
    fig, axes = plt.subplots(2, 2, figsize=(11, 8))
    for name, (_, states) in results.items():
        axes[0, 0].plot(states[:, 0], states[:, 2], lw=0.8, label=name)
        axes[0, 1].plot(states[:, 1], states[:, 3], lw=0.8, label=name)
        axes[1, 0].plot(states[:, 0], states[:, 1], lw=0.8, label=name)
        axes[1, 1].plot(states[:, 2], states[:, 3], lw=0.8, label=name)
    axes[0, 0].set(xlabel="q1", ylabel="p1", title="Phase portrait: q1-p1")
    axes[0, 1].set(xlabel="q2", ylabel="p2", title="Phase portrait: q2-p2")
    axes[1, 0].set(xlabel="q1", ylabel="q2", title="Configuration projection")
    axes[1, 1].set(xlabel="p1", ylabel="p2", title="Momentum projection")
    for ax in axes.ravel():
        ax.grid(alpha=0.3)
    axes[0, 0].legend(loc="best")
    fig.suptitle(title)
    fig.tight_layout()

plot_phase_portraits(results_dt001, "State-space projections, dt=0.01")


## 5.1 不同时间步长下的能量误差

下面比较三种时间步长：

$$
\Delta t=0.01,\quad 0.005,\quad 0.0025.
$$

为了让课堂演示在 CPU 上可以直接运行，下面默认把终止时间设为 `t_end=20.0`。如果你想看更明显的长期差异，可以把 `t_end` 改成 `50.0` 或 `100.0`，但运行时间会增加。

给学生的一句话总结：**减小时间步能改善所有方法，但辛方法的长期误差形态通常更健康。**


In [ ]:
def timestep_study(z0: np.ndarray, dts=(0.01, 0.005, 0.0025), t_end: float = 20.0):
    rows = []
    all_results = {}
    for dt in dts:
        print("\n" + "=" * 70)
        print(f"dt = {dt}")
        results = run_all_methods(z0, t_end=t_end, dt=dt)
        all_results[dt] = results
        for name, (_, states) in results.items():
            e = energy_history(states)
            drift = e - e[0]
            rows.append(
                {
                    "dt": dt,
                    "method": name,
                    "max_abs_drift": float(np.max(np.abs(drift))),
                    "final_abs_drift": float(abs(drift[-1])),
                }
            )
    return rows, all_results

rows, all_dt_results = timestep_study(z0, t_end=20.0)

print("\nsummary")
print(f"{'dt':>8s} | {'method':>20s} | {'max |H-H0|':>14s} | {'final |H-H0|':>14s}")
print("-" * 70)
for row in rows:
    print(f"{row['dt']:8.4f} | {row['method']:>20s} | {row['max_abs_drift']:14.6e} | {row['final_abs_drift']:14.6e}")


In [ ]:
def plot_timestep_summary(rows) -> None:
    methods = list(STEPPERS.keys())
    dts = sorted({row["dt"] for row in rows}, reverse=True)
    plt.figure(figsize=(8, 5))
    for method in methods:
        xs = []
        ys = []
        for dt in dts:
            match = [row for row in rows if row["dt"] == dt and row["method"] == method][0]
            xs.append(dt)
            ys.append(match["max_abs_drift"])
        plt.loglog(xs, ys, marker="o", label=method)
    plt.gca().invert_xaxis()
    plt.xlabel(r"$\Delta t$")
    plt.ylabel(r"max $|H(t)-H(0)|$")
    plt.title("Energy drift vs. timestep")
    plt.grid(alpha=0.3, which="both")
    plt.legend()
    plt.tight_layout()

plot_timestep_summary(rows)


# 6. 如何阅读实验结果？

你可能会观察到：

- RK4 在短时间内可能非常准确；
- Symplectic Euler 因为只有一阶，误差可能明显大一些；
- Störmer-Verlet 和 Implicit Midpoint 通常表现出更稳定的长期能量行为；
- 辛方法的 Hamiltonian 误差不一定最小，但常常呈现有界振荡，而不是长期偏移。

这里要避免一个误解：辛积分器并不是“自动精确守恒能量”。一般来说，它精确保持的是辛结构。根据 backward error analysis，可以把辛方法理解为精确求解了一个与原 Hamiltonian 很接近的修正 Hamiltonian。因此真实 Hamiltonian 的误差通常会围绕零附近振荡。

给学生的一句话总结：**辛方法的卖点不是每一步能量误差最小，而是长期几何行为更可信。**


# 7. 从辛积分到论文：结构保持是一条主线

到这里，本课程两条线已经汇合。

PDE-PINN 线强调：

$$
\mathcal{L}=\mathcal{L}_{data}+\lambda_{PDE}\mathcal{L}_{PDE}+\lambda_{cons}\mathcal{L}_{cons}.
$$

也就是说，把 PDE 和守恒量写进 loss，让神经网络训练时遵守物理规律。

Hamilton 系统线强调：

$$
z_{n+1}=\Phi_{\Delta t}(z_n),
\qquad
(D\Phi_{\Delta t})^T J(D\Phi_{\Delta t})=J.
$$

也就是说，把辛结构写进时间推进算法，让长期轨道遵守 Hamilton 系统的几何性质。

这两者可以统一到一个论文主题中：

> 面向物理系统的结构保持数值学习方法。

可以考虑的小课题方向：

1. **Conservation-PINN 与普通 PINN 对比**：在平流或 Burgers 方程中系统比较质量守恒误差；
2. **Hamilton 系统中的积分器对比**：在双摆或简化振子中比较 RK4、Verlet、Implicit Midpoint 的长期能量行为；
3. **结构保持神经网络**：把 Hamiltonian Neural Network 或 Symplectic Neural Network 作为后续拓展；
4. **混合思路**：用神经网络学习未知 Hamiltonian，再用辛积分器推进，而不是用普通 ODE solver。

一个可发表的小课题不一定要“大而全”。更现实的路线是：选一个简单系统，提出一个清楚的结构保持约束，做扎实的对比实验，并解释为什么它改善了长期物理一致性。

给学生的一句话总结：**论文的关键不只是换模型，而是说明你保住了哪个物理结构，以及这个结构为什么带来更可信的结果。**


# 8. 课后任务

## 必做

1. 跑通本 notebook，记录四种积分器在 `dt=0.01` 时的最大 Hamiltonian 漂移；
2. 将 `t_end` 从 `20.0` 改为 `50.0`，观察 RK4 和辛方法的长期能量误差形态；
3. 分别画出 $(q_1,p_1)$ 和 $(q_2,p_2)$ 相图，并用 3-5 句话解释你看到的差异；
4. 用自己的话解释：为什么“RK4 阶数更高”不等于“长期 Hamilton 系统模拟一定更好”。

## 选做

1. 把固定点迭代替换成 Newton 迭代，比较收敛速度；
2. 选择更剧烈的初值，例如 `z0 = [1.8, -0.7, 0.0, 0.0]`，观察混沌双摆下不同积分器的差异；
3. 在简单谐振子上实现显式 Störmer-Verlet，并验证它的能量误差有界振荡；
4. 阅读 Hamiltonian Neural Network 的基本思想，尝试把“学习 Hamiltonian + 辛积分推进”写成一个小项目计划。

给学生的一句话总结：**第 8 节之后，你已经具备了把 PINN、守恒约束和辛结构组织成一个本科科研小课题的基础。**
